# U-Net Image Segmentation Tutorial
This code demonstrates how to build and train a **U-Net** architecture using TensorFlow/Keras for binary image segmentation.

### Key Concepts:
* **Encoder (Downsampling Path):** Captures context via convolutions and max pooling.
* **Decoder (Upsampling Path):** Enables precise localization using transposed convolutions.
* **Skip Connections:** Concatenates encoder features with decoder features to recover spatial information lost during downsampling.

## 1. Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt

## 2. Define U-Net Building Blocks

In [ ]:
def conv_block(inputs, num_filters):
    """Two 3x3 Convolutions, each followed by Batch Normalization and ReLU."""
    x = layers.Conv2D(num_filters, 3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(num_filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def encoder_block(inputs, num_filters):
    """Convolution block followed by Max Pooling."""
    x = conv_block(inputs, num_filters)
    p = layers.MaxPool2D((2, 2))(x)
    return x, p

def decoder_block(inputs, skip_features, num_filters):
    """Upsampling followed by concatenation with skip connection features."""
    x = layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding='same')(inputs)
    x = layers.Concatenate()([x, skip_features])
    x = conv_block(x, num_filters)
    return x

## 3. Build the U-Net Model

In [ ]:
def build_unet(input_shape):
    inputs = layers.Input(input_shape)

    # Encoder (Downsampling)
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)

    # Bottleneck
    b1 = conv_block(p4, 1024)

    # Decoder (Upsampling)
    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    # Output layer (Binary Segmentation)
    outputs = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(d4)

    model = models.Model(inputs, outputs, name='U-Net')
    return model

# Create and compile the model
model = build_unet((128, 128, 3))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 4. Data Preparation
We generate synthetic 128x128 images with white squares and create the corresponding masks.

In [ ]:
def create_synthetic_data(num_samples=200):
    images = np.zeros((num_samples, 128, 128, 3))
    masks = np.zeros((num_samples, 128, 128, 1))
    for i in range(num_samples):
        x, y = np.random.randint(20, 100, size=2)
        images[i, x:x+20, y:y+20, :] = 1.0
        masks[i, x:x+20, y:y+20, 0] = 1.0
    return images, masks

x_train, y_train = create_synthetic_data(200)

## 5. Training

In [ ]:
print("Training U-Net on synthetic shapes...")
model.fit(x_train, y_train, epochs=5, batch_size=8)

## 6. Results and Visualization

In [ ]:
sample_img = x_train[0:1]
prediction = model.predict(sample_img)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(sample_img[0])
plt.subplot(1, 2, 2)
plt.title("Predicted Mask")
plt.imshow(prediction[0].reshape(128, 128), cmap='gray')
plt.show()